# Career Waze — Dual-Index Recruitment Agent

Este notebook estende o `platform_demo` com dois índices FAISS:

```
OpenAIRuntime  ←  zero mudança ao trocar provider
    ↓
CandidateSearchTool    execute() + schema()   → candidates_index
ExperienceSearchTool   execute() + schema()   → experiences_index
    ↓
CandidateRetriever  /  ExperienceRetriever
    ↓
EmbeddingProvider     OpenAIEmbeddingProvider
VectorStore           FAISSVectorStore  (dois índices, uma instância)

VectorMemoryProvider  (auto-save por turno)
    ↓
EmbeddingProvider  +  VectorStore  (mesmo FAISS, índice separado)

SearchMemoryTool  ←  recupera contexto de turns anteriores
```

**Regra de validação:** trocar qualquer provider muda só o wiring. Nunca a business logic.

## 0. Setup

In [ ]:
!pip install -q -U google-genai google-adk pydantic python-dotenv faiss-cpu requests pyyaml openai

In [1]:
import os
import sys
from pathlib import Path

# Colab: clona o repositório e aponta o sys.path
# Local: apenas adiciona a raiz do repo ao sys.path
try:
    from google.colab import userdata  # só existe no Colab
    if not os.path.exists('/content/agent_workshop'):
        !git clone https://github.com/Willblemos2/agent_workshop.git /content/agent_workshop
    repo_root = '/content/agent_workshop'
except ModuleNotFoundError:
    # Sobe dois níveis a partir de notebooks/workshops/
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

DATA_DIR = Path(repo_root) / 'data'
print(f'repo_root: {repo_root}')
print(f'data_dir : {DATA_DIR}')

repo_root: c:\Users\willi\Documents\Projects\agent_workshop
data_dir : c:\Users\willi\Documents\Projects\agent_workshop\data


In [2]:
# Colab: lê do Secrets (menu lateral → chave 🔑 → GOOGLE_API_KEY / OPENAI_API_KEY)
# Local: lê do .env na raiz do repositório
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('API keys carregadas do Colab Secrets.')
except ModuleNotFoundError:
    from src.shared.config import load_env
    load_env()
    print('API keys carregadas do .env local.')

API keys carregadas do .env local.


## 0c. Provider Selection

Uma linha ativa por bloco. Comente a ativa e descomente a alternativa para trocar.

In [3]:
# ── Embedding Provider — ambos 3072-dim, sem re-indexar ao trocar ──────────
from src.foundation.embeddings import OpenAIEmbeddingProvider
embedding_provider = OpenAIEmbeddingProvider()  # text-embedding-3-large | OPENAI_API_KEY

# from src.foundation.embeddings import GeminiEmbeddingProvider
# embedding_provider = GeminiEmbeddingProvider()  # gemini-embedding-001 | GOOGLE_API_KEY

print(f"Embedding: {type(embedding_provider).__name__}")

Embedding: OpenAIEmbeddingProvider


In [4]:
# ── LLM Provider ─────────────────────────────────────────────────────────────
from src.foundation.llm import OpenAILLMProvider
llm_provider = OpenAILLMProvider()    # gpt-4o-mini | OPENAI_API_KEY

# from src.foundation.llm import GeminiLLMProvider
# llm_provider = GeminiLLMProvider()  # gemini-2.0-flash | GOOGLE_API_KEY

print(f"LLM: {type(llm_provider).__name__}")

LLM: OpenAILLMProvider


In [5]:
# ── Runtime ──────────────────────────────────────────────────────────────────
_INSTRUCTION = (
    "You are an expert recruitment assistant with access to two search indexes. "
    "Use candidate_search to find and rank full candidate profiles by overall fit. "
    "Use experience_search to find specific roles, skills at a company, or career patterns — "
    "then cross-reference the candidate_id with candidate_search for full profile details. "
    "Use search_memory to recall relevant context from past turns before answering."
)

from src.runtime import OpenAIRuntime
def build_runtime(memory_provider=None, user_id="default_user"):
    return OpenAIRuntime(model="gpt-4o-mini", instruction=_INSTRUCTION,
                        memory_provider=memory_provider, user_id=user_id)

# from src.runtime import ADKRuntime
# def build_runtime(memory_provider=None, user_id="default_user"):
#     return ADKRuntime(model="gemini-2.0-flash", instruction=_INSTRUCTION,
#                      memory_provider=memory_provider, user_id=user_id)

print(f"Runtime: {build_runtime().__class__.__name__}")

Runtime: OpenAIRuntime


## 1. Dataset

Carregamos dois datasets com embeddings pré-computados (3072-dim).
Nenhuma chamada de API é feita nesta etapa.

- `candidates_df` — perfis completos com `full_experience` e `embedding`
- `experiences_df` — experiências individuais com `embedded_content`

> Se `experiences.json` não existir, rode `feature_engineering_workshop.ipynb` primeiro.

In [6]:
from src.shared.dataset import load_processed_candidates, load_experiences

candidates_df = load_processed_candidates()
experiences_df = load_experiences()

print(f'Candidatos  : {len(candidates_df)}')
print(f'Experiências: {len(experiences_df)}')
experiences_df[['experience_id', 'candidate_id', 'embedded_content']].head(3)

[dataset] loading from C:\Users\willi\Documents\Projects\agent_workshop\data\processed_candidates.json
[dataset] loading from C:\Users\willi\Documents\Projects\agent_workshop\data\experiences.json
Candidatos  : 2483
Experiências: 11585


,experience_id,candidate_id,embedded_content
0,5f8c8827-861a-48eb-b653-c5efdc260b40,3547447,"[0.01355743408203125, 0.00971221923828125, -0...."
1,a6c5350c-4fc0-471b-a9d6-ecc0d2d3c078,3547447,"[0.000530242919921875, 0.0262298583984375, -0...."
2,836d003b-fca1-47e6-a64b-3e2d4e6a8932,3547447,"[-0.01337432861328125, 0.0120086669921875, -0...."


## 2. Foundation — Vector Store

Uma única instância `FAISSVectorStore` gerencia os dois índices.
Os embeddings já estão no dataset — zero chamadas de API aqui.

In [7]:
from src.foundation.vector_store import FAISSVectorStore, build_records
from src.foundation.vector_store.schemas import CANDIDATE_SCHEMA, EXPERIENCE_SCHEMA

store = FAISSVectorStore()

# ── Candidates index ──────────────────────────────────────────────────────────
store.create_index(index_name='candidates_index', schema=CANDIDATE_SCHEMA)
candidate_records = build_records(
    rows=candidates_df.to_dict(orient='records'),
    id_field='id',
)
store.insert(index_name='candidates_index', records=candidate_records)

# ── Experiences index ─────────────────────────────────────────────────────────
store.create_index(index_name='experiences_index', schema=EXPERIENCE_SCHEMA)
experience_records = build_records(
    rows=experiences_df.to_dict(orient='records'),
    id_field='experience_id',
    embedding_field='embedded_content',
)
store.insert(index_name='experiences_index', records=experience_records)

print(f'{len(candidate_records)} candidatos indexados  → candidates_index')
print(f'{len(experience_records)} experiências indexadas → experiences_index')

2483 candidatos indexados  → candidates_index
11585 experiências indexadas → experiences_index


## 3. Foundation — Embedding Provider

O provider é injetado — os retrievers não sabem qual está sendo usado.

In [8]:
sample = embedding_provider.embed("Senior Python engineer")
print(f"Provider : {type(embedding_provider).__name__}")
print(f"Dimension: {len(sample)}")

Provider : OpenAIEmbeddingProvider
Dimension: 3072


## 4. Application — Retrievers

Dois retrievers, mesma injeção de dependências.
Nenhum deles conhece OpenAI, nenhum deles conhece FAISS.

In [9]:
from src.applications.retrieval import CandidateRetriever, ExperienceRetriever

candidate_retriever = CandidateRetriever(
    vector_store=store,
    embedding_provider=embedding_provider,
)

experience_retriever = ExperienceRetriever(
    vector_store=store,
    embedding_provider=embedding_provider,
)

# ── Teste: candidate_search ───────────────────────────────────────────────────
print('--- candidate_search ---')
for r in candidate_retriever.search('Python developer with machine learning experience', top_k=3):
    print(f'[{r.score:.4f}] {r.first_name} {r.last_name} — {r.content_preview[:80]}...')

print()

# ── Teste: experience_search ──────────────────────────────────────────────────
print('--- experience_search ---')
for r in experience_retriever.search('machine learning engineer at a startup', top_k=3):
    print(f'[{r.score:.4f}] {r.job_title} @ {r.company} (candidate: {r.candidate_id[:8]}...)')

--- candidate_search ---
[0.4277] Riley Thomas — DATA ANALYST       Professional Summary    Industrial and Systems Engineering gr...
[0.4272] Riley Smith — SOFTWARE DEVELOPER         Professional Summary    Enthusiastic computer enginee...
[0.4134] Sam Davis — Highlights          Prog. Languages:          C (5+ yrs), Python (3+ yrs), Java ...

--- experience_search ---
[0.4953] College Student @ University Projects LLC (candidate: 21156767...)
[0.4560] Data Analyst @ Data Analytics Firm (candidate: 18448085...)
[0.4529] Project Intern @ Credit Card Company (candidate: 18448085...)


## 5. Tool Layer

Duas tools, dois contratos idênticos: `execute()` + `schema()`.
O runtime lê `schema()` para registrar cada tool — nunca toca a business logic.

In [10]:
from src.tools import CandidateSearchTool, ExperienceSearchTool

candidate_tool = CandidateSearchTool(retriever=candidate_retriever)
experience_tool = ExperienceSearchTool(retriever=experience_retriever)

for tool in [candidate_tool, experience_tool]:
    s = tool.schema()
    print(f'Tool      : {s.name}')
    print(f'Descrição : {s.description}')
    print(f'Parâmetros:')
    for p in s.parameters:
        req = 'obrigatório' if p.required else 'opcional'
        print(f'  - {p.name} ({p.type}, {req}): {p.description}')
    print()

Tool      : candidate_search
Descrição : Search for candidates matching a natural language query. Returns a ranked list of candidates with profile information.
Parâmetros:
  - query (string, obrigatório): Natural language description of the ideal candidate.
  - top_k (integer, opcional): Maximum number of candidates to return. Defaults to 10.

Tool      : experience_search
Descrição : Search individual work experiences across all candidates. Use this to find specific roles, skills at a company, or career patterns. Returns matched experiences with candidate_id to cross-reference with candidate_search.
Parâmetros:
  - query (string, obrigatório): Natural language description of the experience to find (e.g. 'machine learning engineer at a startup').
  - top_k (integer, opcional): Maximum number of experiences to return. Defaults to 10.



In [11]:
# execute() — chamada direta, sem agent
print('--- candidate_search ---')
result = candidate_tool.execute(query='data scientist with NLP skills', top_k=3)
if result.error:
    print(f'Erro: {result.error}')
else:
    for c in result.content:
        print(f"[{c['score']:.4f}] {c['first_name']} {c['last_name']}")

print()

print('--- experience_search ---')
result = experience_tool.execute(query='software engineer at a fintech company', top_k=3)
if result.error:
    print(f'Erro: {result.error}')
else:
    for e in result.content:
        print(f"[{e['score']:.4f}] {e['job_title']} @ {e['company']}")

--- candidate_search ---
[0.4030] Riley Thomas
[0.4006] Riley Wilson
[0.3766] Taylor Smith

--- experience_search ---
[0.5584] Software Engineer @ Development Firm Ltd.
[0.5303] Software Engineer @ Plausible Tech Solutions
[0.5298] Software Engineer @ Early Stage Software Company


## 6. Runtime — Agent

As duas tools correm dentro do runtime configurado em **0c**.
O runtime lê `tool.schema()` e converte para o formato do provider automaticamente.

Zero mudança nas tools, zero mudança nos retrievers.

In [12]:
runtime = build_runtime()
print(f"Runtime: {type(runtime).__name__}")

response = await runtime.run(
    input="Find me candidates who have worked as data engineers. Also show me their specific data engineering experiences.",
    tools=[candidate_tool, experience_tool],
)
print(response)

Runtime: OpenAIRuntime
[tool call → candidate_search({'query': 'candidates who have worked as data engineers', 'top_k': 10})]
[tool call → experience_search({'query': 'data engineer', 'top_k': 10})]
[tool call → candidate_search({'query': 'data engineer', 'top_k': 10})]
[tool call → experience_search({'query': 'data engineer', 'top_k': 10})]
Here are candidates who have worked as data engineers, along with their relevant experiences:

### Candidates

1. **Riley Clark**
   - **Role:** Teradata Senior Associate Consultant
   - **Summary:** Highly dependable Teradata Database Engineer successful at troubleshooting and debugging. Supportive and enthusiastic team player dedicated to achieving goals.
   
2. **Sam Clark**
   - **Role:** DataStage ETL Developer
   - **Summary:** 6 years of ETL experience and extensive experience in business and systems analysis across multiple platforms using the full software development life cycle.
   
3. **Alex Johnson**
   - **Role:** Sales Engineer
   - *

## 7. Long-term Memory

Cada turno é automaticamente salvo no `VectorMemoryProvider`.
O agent recupera contexto via `SearchMemoryTool`.

**Wiring:**
```
VectorMemoryProvider
    ↓
FAISSVectorStore  +  EmbeddingProvider
```

**Regra:** trocar `FAISSVectorStore` por qualquer outro `VectorStore` não muda nenhuma linha de business logic.

In [13]:
from src.foundation.memory import VectorMemoryProvider
from src.tools import SearchMemoryTool

memory_store = FAISSVectorStore()
memory_provider = VectorMemoryProvider(
    vector_store=memory_store,
    embedding_provider=embedding_provider,
)

# Cache de runtime + memory_tool por user_id.
# Cada usuário tem sua própria instância — garante isolamento de memória.
_user_contexts: dict = {}

def get_user_context(user_id: str) -> dict:
    if user_id not in _user_contexts:
        _user_contexts[user_id] = {
            "runtime": build_runtime(memory_provider=memory_provider, user_id=user_id),
            "memory_tool": SearchMemoryTool(memory_provider=memory_provider, user_id=user_id),
        }
    return _user_contexts[user_id]

ctx = get_user_context("recruiter_1")
RUNTIME_PROVIDER = type(ctx['runtime']).__name__
print(f"Runtime : {RUNTIME_PROVIDER}")
print(f"Session : {ctx['runtime']._session_id}")

Runtime : OpenAIRuntime
Session : 500f2600-4696-415c-ad85-97e9865ffe91


In [14]:
# Inspecionar a memória diretamente
# Troque inspect_user_id para ver memórias de outro usuário
inspect_user_id = "recruiter_1"

memories = memory_provider.search(
    query='candidates experiences search',
    user_id=inspect_user_id,
    top_k=10,
)

print(f'{len(memories)} registros encontrados para [{inspect_user_id}]:\n')
for m in memories:
    preview = m.text[:100].replace('\n', ' ')
    print(f'[{m.role:5s}] {preview}...')

0 registros encontrados para [recruiter_1]:



### Chat Loop com Memória

Mesma interface do chat anterior, agora com memória automática entre turnos
e acesso aos dois índices de busca.

In [15]:
active_user_id = "recruiter_1"

print(f"Career Waze ({RUNTIME_PROVIDER}) — Chat")
print("=" * 45)
print("Tools disponíveis: candidate_search | experience_search | search_memory")
print("Comandos: 'sair' para encerrar | 'user <id>' para trocar de usuário\n")

while True:
    user_input = input("Você: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ("sair", "exit", "quit"):
        print("Encerrando.")
        break

    if user_input.lower().startswith("user "):
        active_user_id = user_input.split(" ", 1)[1].strip()
        print(f"-- usuário: {active_user_id} --\n")
        continue

    print(f"Você: {user_input}")

    ctx = get_user_context(active_user_id)
    tools = [candidate_tool, experience_tool, ctx["memory_tool"]]

    agent_started = False
    async for chunk in ctx["runtime"].stream(user_input, tools=tools):
        if not agent_started:
            print("Agent: ", end="", flush=True)
            agent_started = True
        print(chunk, end="", flush=True)
    print()
    print()

Career Waze (OpenAIRuntime) — Chat
Tools disponíveis: candidate_search | experience_search | search_memory
Comandos: 'sair' para encerrar | 'user <id>' para trocar de usuário

Encerrando.
